In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.transforms as T
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn as nn
import numpy as np
from PIL import Image
from IPython.display import clear_output
import torch.nn.functional as F
import gc

# Полностью очищаем оперативную и видеопамять от старых тензоров
gc.collect()
torch.cuda.empty_cache()
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
if torch.cuda.is_available():
    print(f"Имя GPU: {torch.cuda.get_device_name()}")

Используемое устройство: cuda:0
Имя GPU: Tesla T4


In [6]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset

class KagglePetsDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, target_transform=None):
        self.transform = transform
        self.target_transform = target_transform
        
        # Пути к картинкам и маскам
        self.images_dir = os.path.join(root_dir, "images", "images")
        self.masks_dir = os.path.join(root_dir, "annotations", "annotations", "trimaps")
        
        # Быстро собираем имена всех реально существующих файлов в set (поиск в памяти мгновенный)
        existing_images = set(os.listdir(self.images_dir)) if os.path.exists(self.images_dir) else set()
        existing_masks = set(os.listdir(self.masks_dir)) if os.path.exists(self.masks_dir) else set()
        
        # Определяем файл разбиения
        split_filename = "trainval.txt" if split == 'train' else "test.txt"
        split_file_path = os.path.join(root_dir, "annotations", "annotations", split_filename)
        
        if not os.path.exists(split_file_path):
            raise FileNotFoundError(f"Файл разбиения не найден: {split_file_path}")
            
        self.image_paths = []
        self.mask_paths = []
        
        # Читаем имена файлов строго по официальному списку
        with open(split_file_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                
                # Берём строго первое слово в строке — это чистый ID объекта (например, Abyssinian_1)
                base_name = line.split()[0]
                
                img_name = f"{base_name}.jpg"
                mask_name = f"{base_name}.png"
                
                # Проверяем наличие в set (работает за O(1))
                if img_name in existing_images and mask_name in existing_masks:
                    self.image_paths.append(os.path.join(self.images_dir, img_name))
                    self.mask_paths.append(os.path.join(self.masks_dir, mask_name))
        
        print(f"[{split.upper()}] Официально загружено объектов: {len(self.image_paths)}")

    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx])
        
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
            
        return image, mask


In [7]:
# 2. Определение трансформаций для изображений и масок

# Трансформации для входного изображения (RGB)
transform = T.Compose(
    [
        T.Resize((256, 256)),
        T.ToTensor(),
    ]
)

target_transform = T.Compose([
    T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST), # Важно: NEAREST для масок!
    T.PILToTensor(),
    T.Lambda(lambda x: (x - 1).long())
])

In [8]:
KAGGLE_PATH = "/kaggle/input/datasets/devdgohil/the-oxfordiiit-pet-dataset"

# Инициализация путей займет меньше секунды!
train_dataset = KagglePetsDataset(root_dir=KAGGLE_PATH, split='train', transform=transform, target_transform=target_transform)
valid_dataset = KagglePetsDataset(root_dir=KAGGLE_PATH, split='test', transform=transform, target_transform=target_transform)

# Настройка параллельной фоновой загрузки файлов
# Переключаемся на безопасный размер батча 16, чтобы не было CUDA OOM
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("Лоадеры успешно пересозданы с батчем 16!")

[TRAIN] Официально загружено объектов: 3680
[TEST] Официально загружено объектов: 3669
Лоадеры успешно пересозданы с батчем 16!


In [9]:
# 4. Определение функций обучения и оценки

def train(model) -> float:
    """
    Выполняет один цикл обучения модели на обучающем датасете.
    Возвращает среднюю потерю и среднюю попиксельную точность на обучающей выборке.
    """
    model.train() # Переключаем модель в режим обучения (включает Dropout, BatchNorm и т.д.)

    train_loss = 0
    total_pixels = 0
    correct_pixels = 0

    # Итерируемся по батчам в DataLoader'е
    for x, y in tqdm(train_loader, desc='Train'):
        bs = y.size(0) # Размер текущего батча

        # Перемещаем входные изображения и маски на GPU
        # y.squeeze(1) удаляет лишнюю размерность, если маска была (batch_size, 1, H, W)
        # CrossEntropyLoss ожидает входной тензор shape (N, C, H, W) и целевой тензор (N, H, W)
        x, y = x.to(device), y.squeeze(1).to(device)

        optimizer.zero_grad() # Обнуляем градиенты перед каждым шагом оптимизации

        output = model(x) # Выполняем прямой проход (forward pass)

        # Выход модели (output) имеет размерность (batch_size, num_classes, H, W).
        # Целевая маска (y) имеет размерность (batch_size, H, W).
        # CrossEntropyLoss ожидает, что входной тензор будет (N, C, D1, D2, ...)
        # и целевой тензор будет (N, D1, D2, ...).
        # Мы преобразуем их так, чтобы они соответствовали формату:
        # output.reshape(bs, 3, -1) превращает (bs, 3, H, W) в (bs, 3, H*W)
        # y.reshape(bs, -1) превращает (bs, H, W) в (bs, H*W)
        # Это позволяет CrossEntropyLoss работать с каждым пикселем как с отдельной точкой данных
        # для классификации, но сохраняя батчевую структуру.
        loss = loss_fn(output.reshape(bs, 3, -1), y.reshape(bs, -1))

        train_loss += loss.item() # Добавляем значение потерь к общей сумме

        loss.backward() # Выполняем обратный проход (backward pass) для вычисления градиентов

        optimizer.step() # Делаем шаг оптимизации, обновляя веса модели

        # Рассчитываем попиксельную точность (accuracy)
        # output.max(dim=1) возвращает максимальное значение и его индекс по размерности классов (dim=1)
        _, y_pred = output.max(dim=1) # y_pred имеет предсказанный класс для каждого пикселя
        total_pixels += y.numel() # Общее количество пикселей в батче
        correct_pixels += (y == y_pred).sum().item() # Количество правильно предсказанных пикселей

    train_loss /= len(train_loader) # Средняя потеря за эпоху
    accuracy = correct_pixels / total_pixels # Общая попиксельная точность

    return train_loss, accuracy


@torch.inference_mode() # Отключает вычисление градиентов, ускоряет инференс и экономит память
def evaluate(model, loader) -> tuple[float, float]:
    """
    Оценивает модель на валидационном или тестовом датасете.
    Возвращает среднюю потерю и среднюю попиксельную точность.
    """
    model.eval() # Переключаем модель в режим оценки (отключает Dropout, BatchNorm и т.д.)

    total_loss = 0
    total_pixels = 0
    correct_pixels = 0

    for x, y in tqdm(loader, desc='Evaluation'):
        bs = y.size(0)

        x, y = x.to(device), y.squeeze(1).to(device)

        output = model(x)

        loss = loss_fn(output.reshape(bs, 3, -1), y.reshape(bs, -1))

        total_loss += loss.item()

        _, y_pred = output.max(dim=1)
        total_pixels += y.numel()
        correct_pixels += (y == y_pred).sum().item()

    total_loss /= len(loader)
    accuracy = correct_pixels / total_pixels

    return total_loss, accuracy


def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    train_accuracy: list[float],
    valid_accuracy: list[float],
    title: str
):
    """
    Строит графики потерь и точности для обучения и валидации.
    """
    plt.figure(figsize=(16, 8))
    plt.title(title + ' loss')
    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()
    plt.grid()
    plt.show()

    plt.figure(figsize=(16, 8))
    plt.title(title + ' accuracy')
    plt.plot(train_accuracy, label='Train accuracy')
    plt.plot(valid_accuracy, label='Valid accuracy')
    plt.legend()
    plt.grid()
    plt.show()


@torch.inference_mode()
def visualize(model, batch):
    """
    Визуализирует предсказания модели для батча изображений.
    Показывает оригинальное изображение, истинную маску и предсказанную маску.
    """
    model.eval() # Модель в режиме оценки

    xs, ys = batch # Распаковываем батч на изображения и маски
    
    to_pil = T.ToPILImage() # Вспомогательная функция для преобразования тензора в PIL Image

    for i, (x, y) in enumerate(zip(xs, ys)):
        # x.unsqueeze(0) добавляет размерность батча (из (C, H, W) в (1, C, H, W))
        # .cuda() перемещает тензор на GPU для предсказания
        # .squeeze(0) удаляет размерность батча обратно после предсказания
        # .max(dim=0)[1] берет индекс максимального значения (класс) для каждого пикселя
        prediction = model(x.unsqueeze(0).to(device)).squeeze(0).max(dim=0)[1]

        fig, ax = plt.subplots(1, 3, figsize=(24, 8), facecolor='white')

        # Оригинальное изображение
        ax[0].imshow(to_pil(x.cpu())) # Переносим на CPU для визуализации
        ax[0].axis('off')
        ax[0].set_title('Original image')

        # Истинная маска сегментации
        # y.to(torch.uint8) преобразует метки классов в uint8 для корректной визуализации
        ax[1].imshow(to_pil(y.to(torch.uint8)))
        ax[1].axis('off')
        ax[1].set_title('Segmentation mask (Ground Truth)')

        # Предсказанная маска сегментации
        ax[2].imshow(to_pil(prediction.cpu().to(torch.uint8)))
        ax[2].axis('off')
        ax[2].set_title('Prediction')

        plt.subplots_adjust(wspace=0, hspace=0.1) # Убираем отступы между графиками
        plt.show()

        if i >= 4: # Визуализируем только первые 5 изображений из батча
            break


def whole_train_valid_cycle(model, num_epochs, title):
    """
    Запускает полный цикл обучения и валидации модели.
    """
    train_loss_history, valid_loss_history = [], []
    train_accuracy_history, valid_accuracy_history = [], []

    # Берем один батч из валидационного датасета для регулярной визуализации
    # Это позволяет видеть прогресс модели во время обучения.
    sample_valid_batch = next(iter(valid_loader))

    for epoch in range(num_epochs):
        print(f"--- Эпоха {epoch+1}/{num_epochs} ---")
        train_loss, train_accuracy = train(model)
        valid_loss, valid_accuracy = evaluate(model, valid_loader)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        # clear_output(wait=True) # Очищаем вывод для обновления графиков в реальном времени

        # plot_stats(
        #     train_loss_history, valid_loss_history,
        #     train_accuracy_history, valid_accuracy_history,
        #     title
        # )
        print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
        print(f"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_accuracy:.4f}")

        if valid_accuracy >= 0.90:
            print("Целевая метрика в 90% достигнута!")
            
    
            
            # Сохраняем строго через torch.save на диск в текущую папку
            # Переносим на CPU для гарантированной совместимости с чекером
            save_predictions_for_checker(model, valid_dataset)
            print("Тензор предсказаний успешно записан в файл 'predictions.pt'")
            break
        
        # # Визуализируем предсказания на одном и том же батче валидационной выборки
        # print("\nВизуализация предсказаний на валидационной выборке:")
        # visualize(model, sample_valid_batch)"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_accuracy:.4f}")
@torch.inference_mode()
def save_predictions_for_checker(model, dataset):
    model.eval()
    
    # Генерация индексов строго по условию задачи
    np.random.seed(100)
    idx = np.random.randint(len(dataset), size=200)
    
    collected_preds = []
    
    for i in idx:
        # Берем объект напрямую из датасета
        x, _ = dataset[i]  # x имеет размер (3, 256, 256)
        x = x.unsqueeze(0).to(device)  # Добавляем размерность батча -> (1, 3, 256, 256)
        
        # Получаем логиты от модели -> (1, 3, 256, 256)
        output = model(x)
        
        # Определяем наиболее вероятный класс для каждого пикселя -> (1, 256, 256)
        # Сразу переносим на CPU и освобождаем память GPU
        pred_class = output.max(dim=1)[1].cpu()
        
        # Добавляем размерность канала, чтобы получить (1, 1, 256, 256)
        pred_class = pred_class.unsqueeze(1)
        
        # Переводим в uint8 по требованию NB из задания
        pred_class = pred_class.to(torch.uint8)
        
        collected_preds.append(pred_class)
        
    # Объединяем все 200 тензоров по размерности батча на CPU
    final_tensor = torch.cat(collected_preds, dim=0)  # Результат: [200, 1, 256, 256]
    
    print(f"Итоговый размер тензора: {list(final_tensor.shape)}")
    
    # Сохраняем на диск
    torch.save(final_tensor, 'predictions.pt')
    print("Тензор успешно сохранен в файл 'predictions.pt'!")

In [13]:
# 5. Реализация архитектуры UNet

import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2),
        )

    def forward(self, x):
        return self.block(x)

class UNET(nn.Module):
    def __init__(self):
        super().__init__()
        base_channels = 32  # Уменьшаем с 64 до 32. Сеть станет в 4 раза легче!

        # Энкодер (Down-sampling)
        self.down1 = ConvBlock(3, base_channels)             # 32
        self.down2 = ConvBlock(base_channels, base_channels * 2)     # 64
        self.down3 = ConvBlock(base_channels * 2, base_channels * 4) # 128
        self.down4 = ConvBlock(base_channels * 4, base_channels * 8) # 256
        self.down5 = ConvBlock(base_channels * 8, base_channels * 16) # 512

        # Горлышко (Bottleneck)
        self.bottleneck = ConvBlock(base_channels * 16, base_channels * 32) # 1024

        # Декодер (Up-sampling)
        self.up5 = ConvBlock(base_channels * 32 + base_channels * 16, base_channels * 16)
        self.up4 = ConvBlock(base_channels * 16 + base_channels * 8, base_channels * 8)
        self.up3 = ConvBlock(base_channels * 8 + base_channels * 4, base_channels * 4)
        self.up2 = ConvBlock(base_channels * 4 + base_channels * 2, base_channels * 2)
        self.up1 = ConvBlock(base_channels * 2 + base_channels, base_channels)

        self.out = nn.Conv2d(in_channels=base_channels, out_channels=3, kernel_size=1)
        self.downsample = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # --- Энкодер ---
        res1 = self.down1(x)
        x = self.downsample(res1)

        res2 = self.down2(x)
        x = self.downsample(res2)

        res3 = self.down3(x)
        x = self.downsample(res3)

        res4 = self.down4(x)
        x = self.downsample(res4)
        
        res5 = self.down5(x)
        x = self.downsample(res5)

        # --- Горлышко ---
        x = self.bottleneck(x)

        # --- Декодер ---
        # 5 уровень
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat((x, res5), dim=1)
        x = self.up5(x)

        # 4 уровень
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat((x, res4), dim=1)
        x = self.up4(x)

        # 3 уровень
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat((x, res3), dim=1)
        x = self.up3(x)

        # 2 уровень
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat((x, res2), dim=1)
        x = self.up2(x)

        # 1 уровень
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat((x, res1), dim=1)
        x = self.up1(x)

        return self.out(x)


In [11]:
# 6. Инициализация модели, функции потерь и оптимизатора

# Функция потерь: CrossEntropyLoss для многоклассовой классификации (попиксельной).
# Она автоматически применяет softmax к выходам модели и затем вычисляет кросс-энтропию.
loss_fn = nn.CrossEntropyLoss()

# Создаем экземпляр UNet модели и перемещаем ее на GPU.
model = UNET().to(device)

# Оптимизатор Adam: популярный выбор для нейронных сетей.
# lr (learning rate): Гиперпараметр, контролирующий размер шага при обновлении весов.
# 1e-3 (0.001) - хорошее начальное значение.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [14]:
# 7. Запуск цикла обучения

print("\n--- Запуск обучения UNet для семантической сегментации ---")
# num_epochs (количество эпох): Гиперпараметр, определяющий, сколько раз модель
# будет проходить по всему обучающему датасету.
# Чем больше эпох, тем лучше модель может обучиться, но выше риск переобучения.
whole_train_valid_cycle(model, num_epochs=60, title='UNET segmentation')


--- Запуск обучения UNet для семантической сегментации ---
--- Эпоха 1/60 ---


Train:   0%|          | 0/230 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 46.81 MiB is free. Process 49 has 10.08 GiB memory in use. Including non-PyTorch memory, this process has 4.44 GiB memory in use. Of the allocated memory 4.22 GiB is allocated by PyTorch, and 96.11 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)